In [1]:
library(dplyr)
library(Seurat)
library(ggplot2)
library(patchwork)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.7.5 but the current
version is 1.7.6; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


v5.6 adds significant improvements in speed and efficiency, particularly for large datasets.
For more information, see https://satijalab.org/seurat/articles/announcements.




In [2]:
prechemo <- readRDS('../Results/PreChemo/prechemo_annotated.rds')

postchemo <- readRDS('../Results/PostChemo/postchemo_annotated.rds')

In [4]:
head(prechemo@meta.data)

,orig.ident,nCount_RNA,nFeature_RNA,sample_id,percent.mt,percent.eryth,treatment,integrated_snn_res.0.3,seurat_clusters,cell_type
,<chr>,<dbl>,<int>,<chr>,<dbl>,<dbl>,<chr>,<fct>,<fct>,<chr>
AAACCCAAGGTGGCTA-1_1,GSM6204109_P01,2553,1205,GSM6204109_P01,6.227967,0.00000000,Untreated,4,4,Gamma-delta/NK-like T
AAACCCAAGTTTCGGT-1_1,GSM6204109_P01,2959,1084,GSM6204109_P01,5.846570,0.00000000,Untreated,2,2,CD4-like T cells
AAACCCACAACCCTAA-1_1,GSM6204109_P01,16327,4713,GSM6204109_P01,4.832486,0.00000000,Untreated,0,0,Epithelial
AAACCCAGTATCGATC-1_1,GSM6204109_P01,3951,2006,GSM6204109_P01,4.910149,0.00000000,Untreated,9,9,Endothelial
AAACCCAGTCAAACGG-1_1,GSM6204109_P01,28977,6504,GSM6204109_P01,5.003969,0.00000000,Untreated,0,0,Epithelial
AAACCCATCATCGCTC-1_1,GSM6204109_P01,5654,1950,GSM6204109_P01,2.670676,0.01768659,Untreated,1,1,SPP1+ Macrophages


In [5]:
colnames(prechemo@meta.data)

[1] "orig.ident"             "nCount_RNA"             "nFeature_RNA"          
 [4] "sample_id"              "percent.mt"             "percent.eryth"         
 [7] "treatment"              "integrated_snn_res.0.3" "seurat_clusters"       
[10] "cell_type"

In [6]:
colnames(postchemo@meta.data)

[1] "orig.ident"             "nCount_RNA"             "nFeature_RNA"          
 [4] "sample_id"              "percent.mt"             "percent.eryth"         
 [7] "treatment"              "integrated_snn_res.0.3" "seurat_clusters"       
[10] "cell_type"

In [7]:
Assays(prechemo)
Assays(postchemo)

[1] "RNA"        "integrated"

[1] "RNA"        "integrated"

In [8]:
DefaultAssay(prechemo) <- "RNA"
DefaultAssay(postchemo) <- "RNA"

In [9]:
combined <- merge(prechemo, postchemo, add.cell.ids = c("PreChemo", "PostChemo"), project="PDAC", merge.data=FALSE)

In [10]:
DefaultAssay(combined) <- "RNA"

In [11]:
table(combined$treatment)


  Treated Untreated 
    15252      8020 

In [12]:
table(combined$sample_id, combined$treatment)

                
                 Treated Untreated
  GSM6204109_P01       0      4680
  GSM6204110_P02       0      3340
  GSM6204111_P03   14142         0
  GSM6204114_P06    1110         0

In [13]:
table(combined$cell_type, combined$treatment)

                             
                              Treated Untreated
  Acinar cells                     23         0
  B cells                           0        52
  CAFs                           2188         0
  CD4-like T cells                  0      1270
  CD8+ T cells                      0      1056
  cDC2                              0        86
  Endocrine cells                  49         0
  Endothelial                     370        68
  Epithelial                        0      2488
  Epithelial cells               7103         0
  Fibroblast/CAF-like               0        40
  Gamma-delta/NK-like T             0       425
  Inflammatory Monocytes          493         0
  LYVE1+ Macrophages               53         0
  Mast cells                      180         0
  NK cells                          0       347
  Osteoclast-like Macrophages      43         0
  pDC                               0        19
  Pericytes/VSMCs                 214         0
  Plasma c

In [14]:
combined$cell_type_broad <- as.character(combined$cell_type)

combined$cell_type_broad[
  combined$cell_type %in% c(
    "Epithelial",
    "Epithelial cells",
    "Proliferating Epithelial"
  )
] <- "Epithelial"

combined$cell_type_broad[
  combined$cell_type %in% c(
    "CAFs",
    "Fibroblast/CAF-like"
  )
] <- "Fibroblasts/CAFs"

combined$cell_type_broad[
  combined$cell_type %in% c(
    "SPP1+ Macrophages",
    "TREM2+ Macrophages",
    "LYVE1+ Macrophages",
    "Osteoclast-like Macrophages",
    "Inflammatory Monocytes"
  )
] <- "Myeloid"

combined$cell_type_broad[
  combined$cell_type %in% c(
    "CD4-like T cells",
    "CD8+ T cells",
    "Tregs",
    "NK cells",
    "Gamma-delta/NK-like T",
    "T/NK Lymphocytes",
    "Proliferating T cells"
  )
] <- "T/NK cells"

In [15]:
table(combined$cell_type_broad, combined$treatment)

                  
                   Treated Untreated
  Acinar cells          23         0
  B cells                0        52
  cDC2                   0        86
  Endocrine cells       49         0
  Endothelial          370        68
  Epithelial          7103      2866
  Fibroblasts/CAFs    2188        40
  Mast cells           180         0
  Myeloid             3327      1572
  pDC                    0        19
  Pericytes/VSMCs      214         0
  Plasma cells         325        54
  T/NK cells          1473      3263

In [16]:
table(
  combined$cell_type_broad,
  combined$sample_id
)

                  
                   GSM6204109_P01 GSM6204110_P02 GSM6204111_P03 GSM6204114_P06
  Acinar cells                  0              0              7             16
  B cells                      11             41              0              0
  cDC2                         51             35              0              0
  Endocrine cells               0              0             49              0
  Endothelial                  63              5            348             22
  Epithelial                 1898            968           6945            158
  Fibroblasts/CAFs             34              6           2144             44
  Mast cells                    0              0            175              5
  Myeloid                    1384            188           3139            188
  pDC                           8             11              0              0
  Pericytes/VSMCs               0              0            207              7
  Plasma cells                 26

In [17]:
tnk <- subset(
  combined,
  subset = cell_type_broad == "T/NK cells"
)

In [18]:
tnk <- JoinLayers(tnk, assay = "RNA")

In [19]:
pseudo_tnk <- AggregateExpression(
  object = tnk,
  assays = "RNA",
  group.by = "sample_id",
  return.seurat = FALSE
)$RNA

Names of identity class contain underscores ('_'), replacing with dashes ('-')
This message is displayed once every 8 hours.


In [20]:
sample_metadata <- unique(
  tnk@meta.data[, c("sample_id", "treatment")]
)

rownames(sample_metadata) <- sample_metadata$sample_id

In [21]:
sample_metadata <- sample_metadata[
  colnames(pseudo_tnk), ,
  drop = FALSE
]

In [22]:
sample_metadata$treatment <- factor(
  sample_metadata$treatment,
  levels = c("Untreated", "Treated")
)

In [26]:
sample_metadata <- unique(
  tnk@meta.data[, c("sample_id", "treatment")]
)

sample_metadata <- as.data.frame(sample_metadata)

rownames(sample_metadata) <- sample_metadata$sample_id

# Reorder to match pseudobulk columns
sample_metadata <- sample_metadata[
  colnames(pseudo_tnk),
  ,
  drop = FALSE
]

sample_metadata$treatment <- factor(
  sample_metadata$treatment,
  levels = c("Untreated", "Treated")
)

In [27]:
sample_metadata

identical(
  rownames(sample_metadata),
  colnames(pseudo_tnk)
)

anyNA(sample_metadata$treatment)

,sample_id,treatment
,<chr>,<fct>
NA,NA,NA
NA.1,NA,NA
NA.2,NA,NA
NA.3,NA,NA


[1] FALSE

[1] TRUE